# 🔍 環境確認 — ARC-AGI-3 Kaggle互換環境チェック

このノートブックは、開発環境が **Kaggle提出環境と互換** であることを確認します。

以下を検証します:
1. Python バージョン (3.12+)
2. GPU / CUDA の利用可能性
3. 主要パッケージのバージョン
4. ARC-AGI-3 エージェントモジュールの import 確認

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 12), f"Python 3.12+ required, got {sys.version}"
print("✅ Python version OK")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print("✅ GPU OK")
else:
    print("⚠️ GPU not available (CPU mode)")

In [ ]:
import numpy as np
import scipy
import pydantic

packages = {
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pydantic": pydantic.__version__,
    "torch": torch.__version__,
}

print("📦 Package Versions:")
for name, version in packages.items():
    print(f"  {name}: {version}")
print("\n✅ Core packages OK")

In [ ]:
# ARC-AGI-3 エージェントモジュールの import 確認
try:
    from acr_agi3.agent.orchestrator import ARCOrchestrator
    from acr_agi3.dsl import primitives
    from acr_agi3.eval.harness import BenchmarkHarness
    from acr_agi3.eval.metrics import exact_match, compute_pass_at_k
    from acr_agi3.submission.entrypoint import run_submission
    print("✅ All acr_agi3 modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Run: pip install -e '.[dev]' in the project root")

In [ ]:
# DSL プリミティブの動作確認
grid = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Original grid:")
print(grid)
print("\nRotated 90°:")
print(primitives.rot90(grid))
print("\nFlipped LR:")
print(primitives.fliplr(grid))
print("\n✅ DSL primitives OK")

In [ ]:
# Orchestrator による簡易推論テスト
orchestrator = ARCOrchestrator()

# 恒等変換 (identity) のサンプルタスク
train_pairs = [
    {"input": np.array([[1, 0], [0, 1]]), "output": np.array([[1, 0], [0, 1]])},
]
test_input = np.array([[2, 0], [0, 2]])

predictions = orchestrator.solve(train_pairs, test_input, max_attempts=2)
print(f"Predictions: {len(predictions)} candidates")
for i, pred in enumerate(predictions):
    print(f"  Attempt {i+1}:")
    print(f"  {pred}")
print("\n✅ Orchestrator OK")

---
## Summary

すべてのチェックが ✅ であれば、この環境は:
- **Kaggle提出環境と互換** (Python 3.12, PyTorch, CUDA)
- **EDD Agent の自律開発が可能** (全モジュールがインポート可能)
- **JupyterLab での実験が可能** (このノートブックが動作)

次のステップ:
1. `notebooks/01_data_exploration.ipynb` — ARC-AGI-3 データセットの探索
2. `notebooks/02_agent_test.ipynb` — エージェントの動作テスト
3. `docker exec arc-agi3-dev pytest -v` — テストスイートの実行